In [ ]:
%%sql

-- Granularidad:
--   1 detección NASA × 1 hora de forecast Weather
-- NASA:
--   detecciones operativas de las últimas 24 horas
-- WEATHER:
--   presente + futuro
-- Espacio:
--   punto Weather más cercano <= 25 km

DROP TABLE IF EXISTS gold_fire_risk;

--creacion de la estructura final de la tabla gold para evaluacion de riesgo de incendios
CREATE TABLE gold_fire_risk (

    fire_id STRING,
    cluster_id STRING,
    fire_detection_timestamp TIMESTAMP,
    fire_latitude DOUBLE,
    fire_longitude DOUBLE,
    forecast_timestamp TIMESTAMP,
    weather_latitude DOUBLE,
    weather_longitude DOUBLE,
    weather_distance_km DOUBLE,
    temperature_2m DOUBLE,
    relative_humidity_2m DOUBLE,
    precipitation DOUBLE,
    wind_speed_10m DOUBLE,
    brightness DOUBLE,
    frp DOUBLE,
    temperature_score INT,
    humidity_score INT,
    wind_score INT,
    precipitation_score INT,
    fire_intensity_score INT,
    fire_risk_score INT,
    fire_risk_level STRING,
    source_fire STRING,
    source_weather STRING,
    updated_at TIMESTAMP
);


--NASA ultimas 24h

--vista temporal para filtrar las alertas termicas activas dentro del marco operativo reciente
CREATE OR REPLACE TEMP VIEW recent_fires AS

SELECT
    SHA2(
        CONCAT_WS(
            '|',
            CAST(latitude AS STRING),
            CAST(longitude AS STRING),
            CAST(fire_detection_timestamp AS STRING)
        ),
        256
    ) AS fire_id,
    cluster_id,
    fire_detection_timestamp,
    CAST(latitude AS DOUBLE) AS fire_latitude,
    CAST(longitude AS DOUBLE) AS fire_longitude,
    CAST(bright_ti4 AS DOUBLE) AS brightness,
    CAST(fire_radiative_power AS DOUBLE) AS frp,
    landing_source_file AS source_fire,
    ingestion_timestamp AS fire_ingestion_timestamp

FROM silver_nasa_fires

WHERE fire_detection_timestamp IS NOT NULL
  AND latitude IS NOT NULL
  AND longitude IS NOT NULL
  AND fire_detection_timestamp >=
        current_timestamp() - INTERVAL 24 HOURS
  AND fire_detection_timestamp <=
        current_timestamp();


--Weather presente + futuro

--he aislado los datos meteorologicos correspondientes a predicciones presentes y futuras
CREATE OR REPLACE TEMP VIEW future_weather AS

SELECT

    CAST(latitude AS DOUBLE) AS weather_latitude,
    CAST(longitude AS DOUBLE) AS weather_longitude,
    forecast_timestamp,
    CAST(temperature_celsius AS DOUBLE) AS temperature_2m,
    CAST(humidity_percentage AS DOUBLE) AS relative_humidity_2m,
    CAST(precipitation_mm AS DOUBLE) AS precipitation,
    CAST(wind_speed_kmh AS DOUBLE) AS wind_speed_10m,
    ingestion_timestamp AS weather_ingestion_timestamp,
    landing_source_file AS source_weather

FROM silver_weather

WHERE forecast_timestamp IS NOT NULL
  AND latitude IS NOT NULL
  AND longitude IS NOT NULL
  AND forecast_timestamp >= current_timestamp();


--Candidatos espaciales

--cruzado espacial mediante la formula de haversine previa acotacion por caja delimitadora
CREATE OR REPLACE TEMP VIEW weather_candidates AS

SELECT

    f.fire_id,
    f.cluster_id,
    f.fire_detection_timestamp,
    f.fire_latitude,
    f.fire_longitude,
    f.brightness,
    f.frp,
    f.source_fire,
    f.fire_ingestion_timestamp,
    w.weather_latitude,
    w.weather_longitude,
    w.forecast_timestamp,
    w.temperature_2m,
    w.relative_humidity_2m,
    w.precipitation,
    w.wind_speed_10m,
    w.source_weather,
    w.weather_ingestion_timestamp,
    (
        6371.0 * 2.0 * ASIN(
            SQRT(
                POWER(
                    SIN(
                        RADIANS(
                            w.weather_latitude
                            - f.fire_latitude
                        ) / 2.0
                    ),
                    2
                )
                +
                COS(
                    RADIANS(f.fire_latitude)
                )
                *
                COS(
                    RADIANS(w.weather_latitude)
                )
                *
                POWER(
                    SIN(
                        RADIANS(
                            w.weather_longitude
                            - f.fire_longitude
                        ) / 2.0
                    ),
                    2
                )
            )
        )
    ) AS weather_distance_km

FROM recent_fires f

INNER JOIN future_weather w

    ON w.weather_latitude BETWEEN
        f.fire_latitude - 0.25
        AND
        f.fire_latitude + 0.25
   AND w.weather_longitude BETWEEN
        f.fire_longitude - 0.35
        AND
        f.fire_longitude + 0.35;


--Weather más cercano por hora

--se selecciona la estacion o punto de prediccion meteorologica mas cercano para cada franja horaria
CREATE OR REPLACE TEMP VIEW nearest_weather AS

SELECT *

FROM (

    SELECT
        wc.*,
        ROW_NUMBER() OVER (
            PARTITION BY
                fire_id,
                fire_detection_timestamp,
                forecast_timestamp
            ORDER BY
                weather_distance_km ASC,
                weather_ingestion_timestamp DESC
        ) AS rn

    FROM weather_candidates wc

    WHERE weather_distance_km <= 25.0
)

WHERE rn = 1;


--Scoring

--vista para asignar puntuaciones parciales segun variables meteorologicas e intensidad
CREATE OR REPLACE TEMP VIEW gold_fire_risk_source AS

SELECT

    fire_id,
    cluster_id,
    fire_detection_timestamp,
    fire_latitude,
    fire_longitude,
    forecast_timestamp,
    weather_latitude,
    weather_longitude,
    weather_distance_km,
    temperature_2m,
    relative_humidity_2m,
    precipitation,
    wind_speed_10m,
    brightness,
    frp,
    CASE
        WHEN temperature_2m >= 35 THEN 3
        WHEN temperature_2m >= 30 THEN 2
        WHEN temperature_2m >= 25 THEN 1
        ELSE 0
    END AS temperature_score,
    CASE
        WHEN relative_humidity_2m < 20 THEN 3
        WHEN relative_humidity_2m < 30 THEN 2
        WHEN relative_humidity_2m < 40 THEN 1
        ELSE 0
    END AS humidity_score,
    CASE
        WHEN wind_speed_10m >= 40 THEN 3
        WHEN wind_speed_10m >= 25 THEN 2
        WHEN wind_speed_10m >= 15 THEN 1
        ELSE 0
    END AS wind_score,
    CASE
        WHEN COALESCE(precipitation, 0) = 0 THEN 2
        WHEN precipitation < 1 THEN 1
        ELSE 0
    END AS precipitation_score,
    CASE
        WHEN frp >= 100 THEN 3
        WHEN frp >= 50 THEN 2
        WHEN frp >= 10 THEN 1
        ELSE 0
    END AS fire_intensity_score,
    source_fire,
    source_weather

FROM nearest_weather;

--Riesgo final

--calculo del indice de riesgo agregado y categorizacion del nivel de peligro
CREATE OR REPLACE TEMP VIEW gold_fire_risk_final AS

SELECT

    fire_id,
    cluster_id,
    fire_detection_timestamp,
    fire_latitude,
    fire_longitude,
    forecast_timestamp,
    weather_latitude,
    weather_longitude,
    weather_distance_km,
    temperature_2m,
    relative_humidity_2m,
    precipitation,
    wind_speed_10m,
    brightness,
    frp,
    temperature_score,
    humidity_score,
    wind_score,
    precipitation_score,
    fire_intensity_score,
    (
        temperature_score
        + humidity_score
        + wind_score
        + precipitation_score
        + fire_intensity_score
    ) AS fire_risk_score,
    CASE
        WHEN (
            temperature_score
            + humidity_score
            + wind_score
            + precipitation_score
            + fire_intensity_score
        ) >= 12
            THEN 'CRITICAL'
        WHEN (
            temperature_score
            + humidity_score
            + wind_score
            + precipitation_score
            + fire_intensity_score
        ) >= 9
            THEN 'HIGH'
        WHEN (
            temperature_score
            + humidity_score
            + wind_score
            + precipitation_score
            + fire_intensity_score
        ) >= 5
            THEN 'MEDIUM'
        ELSE 'LOW'
    END AS fire_risk_level,
    source_fire,
    source_weather,
    current_timestamp() AS updated_at

FROM gold_fire_risk_source;


--INSERT

--insercion masiva del conjunto de datos calculado en la tabla destino gold_fire_risk
INSERT INTO gold_fire_risk

SELECT *

FROM gold_fire_risk_final;